<a href="https://colab.research.google.com/github/srani-1601/Multimodel_search_and_RAG/blob/main/Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from google.colab import files
uploaded = files.upload()

Saving Similarity_score_baseine.xlsx to Similarity_score_baseine.xlsx


In [4]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from tqdm import tqdm # For progress bars

# === Configuration ===
# Set to True if a GPU is available for training, otherwise False
USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")
os.environ["CUDA_VISIBLE_DEVICES"] = "0" if USE_CUDA else ""

# Training Hyperparameters
EPOCHS = 10
BATCH_SIZE = 16 # Batch size for the training loop
LEARNING_RATE = 1e-4
# Batch size for the initial embedding generation to prevent crashes
PROCESSING_BATCH_SIZE = 5
# New: Control the total number of sentences to process. Set to None to process all.
MAX_SENTENCES = 10

print(f"Using device: {DEVICE}")

# === Step 1: Set Up Models ===
print("Loading models...")
# Model 1: Sanskrit ByT5 (for student embeddings)
byt5_model_name = "chronbmm/sanskrit-byt5-dp"
byt5_tokenizer = AutoTokenizer.from_pretrained(byt5_model_name)
byt5_model = AutoModel.from_pretrained(byt5_model_name).to(DEVICE)
byt5_model.eval()

# Model 2: Vyakyarth (for teacher embeddings)
vyakyarth_model = SentenceTransformer("krutrim-ai-labs/vyakyarth")
vyakyarth_model.to(DEVICE)


# === Step 2: Load Sentences and Generate Pre-computed Embeddings in Batches ===

# Load all sentences from your file first
sn_file = "dev.sn"
sentences = []
try:
    with open(sn_file, "r", encoding="utf-8") as f:
        for line in f:
            stripped = line.strip()
            if stripped:
                sentence = stripped.split("\t")[0]
                sentences.append(sentence)
    print(f"Loaded {len(sentences)} sentences from file.")
except FileNotFoundError:
    print(f"Error: The file '{sn_file}' was not found.")
    print("Using dummy sentences for demonstration purposes.")
    sentences = [
        "अहम् आपणम् गच्छामि।", "सः पुस्तकम् पठति।", "बालकाः कन्दुकेन क्रीडन्ति।", "सूर्यः पूर्वस्याम् दिशि उदेति।",
        "अहम् आपणम् गच्छामि।", "सः पुस्तकम् पठति।", "बालकाः कन्दुकेन क्रीडन्ति।", "सूर्यः पूर्वस्याम् दिशि उदेति।",
        "अहम् आपणम् गच्छामि।", "सः पुस्तकम् पठति।", "बालकाः कन्दुकेन क्रीडन्ति।", "सूर्यः पूर्वस्याम् दिशि उदेति।",
        "अहम् आपणम् गच्छामि।", "सः पुस्तकम् पठति।", "बालकाः कन्दुकेन क्रीडन्ति।", "सूर्यः पूर्वस्याम् दिशि उदेति।"
    ]

# --- NEW: Limit the number of sentences to process ---
if MAX_SENTENCES is not None and len(sentences) > MAX_SENTENCES:
    sentences = sentences[:MAX_SENTENCES]
    print(f"Processing the first {len(sentences)} sentences as per MAX_SENTENCES setting.")

# --- Process in batches to save memory ---
all_vyakyarth_embeddings = []
all_byt5_embeddings = []

print(f"\nGenerating embeddings in batches of {PROCESSING_BATCH_SIZE} to conserve memory...")

# Create a loop that processes the sentences in chunks
for i in tqdm(range(0, len(sentences), PROCESSING_BATCH_SIZE), desc="Processing All Sentences"):
    # Get a batch of sentences
    batch_sentences = sentences[i:i + PROCESSING_BATCH_SIZE]

    # 1. Generate Teacher Embeddings for the batch
    batch_embeddings_vyakyarth = vyakyarth_model.encode(
        batch_sentences,
        convert_to_tensor=True,
        show_progress_bar=False, # Disable inner progress bar
        device=DEVICE
    )
    all_vyakyarth_embeddings.append(batch_embeddings_vyakyarth)

    # 2. Generate Student Embeddings for the batch
    batch_embeddings_byt5_list = []
    with torch.no_grad():
        for sentence in batch_sentences:
            inputs = byt5_tokenizer(sentence, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
            output = byt5_model.encoder(**inputs).last_hidden_state

            attention_mask = inputs['attention_mask']
            mask_expanded = attention_mask.unsqueeze(-1).expand(output.size()).float()
            sum_embeddings = torch.sum(output * mask_expanded, 1)
            sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
            pooled = sum_embeddings / sum_mask

            batch_embeddings_byt5_list.append(pooled)

    all_byt5_embeddings.append(torch.cat(batch_embeddings_byt5_list, dim=0))

# Concatenate all the generated batch embeddings into single tensors
embeddings_vyakyarth = torch.cat(all_vyakyarth_embeddings, dim=0)
embeddings_byt5 = torch.cat(all_byt5_embeddings, dim=0)

print("\nFinished generating all embeddings.")
print(f"Shape of Vyakyarth embeddings: {embeddings_vyakyarth.shape}")
print(f"Shape of ByT5 embeddings: {embeddings_byt5.shape}")

# Move embeddings to CPU for use in the dataset
embeddings_vyakyarth_cpu = embeddings_vyakyarth.cpu()
embeddings_byt5_cpu = embeddings_byt5.cpu()


# === Step 3: Define the Distillation Model ===

class DistillationModel(nn.Module):
    """
    This model does NOT contain the original encoder. It only contains
    a projection head that takes pre-computed student embeddings as input
    and learns to map them to the teacher's embedding space.
    """
    def __init__(self, student_embedding_dim, projection_dim):
        super().__init__()
        # This simple network will be trained to perform the mapping.
        self.projection_head = nn.Sequential(
            nn.Linear(student_embedding_dim, student_embedding_dim),
            nn.ReLU(),
            nn.Linear(student_embedding_dim, projection_dim)
        )

    def forward(self, student_embedding):
        # Pass the pre-computed student embedding through the projection head
        distilled_embedding = self.projection_head(student_embedding)
        return distilled_embedding

# === Step 4: Create a Custom Dataset and DataLoader ===

class DistillationDataset(Dataset):
    """
    A PyTorch dataset that provides pairs of pre-computed
    student and teacher embeddings.
    """
    def __init__(self, student_embeddings, teacher_embeddings):
        self.student_embeddings = student_embeddings
        self.teacher_embeddings = teacher_embeddings

    def __len__(self):
        return len(self.student_embeddings)

    def __getitem__(self, idx):
        return self.student_embeddings[idx], self.teacher_embeddings[idx]

# === Step 5: Set Up and Run the Training Loop ===

# Get the embedding dimensions for the model
byt5_embedding_dim = embeddings_byt5.shape[1]
vyakyarth_embedding_dim = embeddings_vyakyarth.shape[1]

# Instantiate our student model
student_distillation_model = DistillationModel(
    student_embedding_dim=byt5_embedding_dim,
    projection_dim=vyakyarth_embedding_dim
).to(DEVICE)

# Create the dataset and dataloader using the pre-computed embeddings
train_dataset = DistillationDataset(embeddings_byt5_cpu, embeddings_vyakyarth_cpu)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Define the loss function and optimizer
loss_function = nn.MSELoss()
optimizer = optim.AdamW(student_distillation_model.parameters(), lr=LEARNING_RATE)

print("\nStarting knowledge distillation training...")
student_distillation_model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for student_embs, teacher_embs in progress_bar:
        # Move data to the selected device
        student_embs = student_embs.to(DEVICE)
        teacher_embs = teacher_embs.to(DEVICE)

        optimizer.zero_grad()

        # Forward pass with the pre-computed student embedding
        distilled_embeddings = student_distillation_model(student_embs)

        loss = loss_function(distilled_embeddings, teacher_embs)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1}/{EPOCHS} - Average Loss: {avg_loss:.4f}")

print("\nTraining complete!")

# === Step 6: Generate New Embeddings with the Trained Model ===

print("\nGenerating new, distilled embeddings for all sentences...")
student_distillation_model.eval()
new_combined_embeddings = []

# Use a dataloader for efficient batch processing
inference_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)

with torch.no_grad():
    for student_embs, _ in tqdm(inference_dataloader, desc="Generating Embeddings"):
        student_embs = student_embs.to(DEVICE)

        distilled_embeddings = student_distillation_model(student_embs)
        new_combined_embeddings.extend(distilled_embeddings.cpu().numpy())

new_combined_embeddings = np.array(new_combined_embeddings)

print(f"\nSuccessfully generated {len(new_combined_embeddings)} new embeddings.")
print(f"Shape of the new embedding matrix: {new_combined_embeddings.shape}")

# You can now save or use `new_combined_embeddings` for downstream tasks.
np.save("distilled_sanskrit_embeddings.npy", new_combined_embeddings)
print("New embeddings saved to distilled_sanskrit_embeddings.npy")


Using device: cpu
Loading models...
Loaded 6148 sentences from file.
Processing the first 10 sentences as per MAX_SENTENCES setting.

Generating embeddings in batches of 5 to conserve memory...


Processing All Sentences: 100%|██████████| 2/2 [00:53<00:00, 26.73s/it]



Finished generating all embeddings.
Shape of Vyakyarth embeddings: torch.Size([10, 768])
Shape of ByT5 embeddings: torch.Size([10, 1536])

Starting knowledge distillation training...


Epoch 1/10: 100%|██████████| 1/1 [00:00<00:00,  5.85it/s, loss=0.0948]


Epoch 1/10 - Average Loss: 0.0948


...

Epoch 9/10 - Average Loss: 0.0870


Epoch 10/10: 100%|██████████| 1/1 [00:00<00:00, 15.67it/s, loss=0.0858]


Epoch 10/10 - Average Loss: 0.0858

Training complete!

Generating new, distilled embeddings for all sentences...


Generating Embeddings: 100%|██████████| 1/1 [00:00<00:00, 144.85it/s]


Successfully generated 10 new embeddings.
Shape of the new embedding matrix: (10, 768)
New embeddings saved to distilled_sanskrit_embeddings.npy


In [13]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm # For progress bars

# === Configuration ===
# Set to True if a GPU is available for training, otherwise False
USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")
os.environ["CUDA_VISIBLE_DEVICES"] = "0" if USE_CUDA else ""

# Training Hyperparameters
EPOCHS = 10
BATCH_SIZE = 16 # Batch size for the training loop
LEARNING_RATE = 1e-4
# Batch size for the initial embedding generation to prevent crashes
PROCESSING_BATCH_SIZE = 10
# Control the total number of sentences to process. Set to None to process all.
MAX_SENTENCES = 50

print(f"Using device: {DEVICE}")

# === Step 1: Set Up Models ===
print("Loading models...")
# Model 1: Sanskrit ByT5 (for student embeddings)
byt5_model_name = "chronbmm/sanskrit-byt5-dp"
byt5_tokenizer = AutoTokenizer.from_pretrained(byt5_model_name)
byt5_model = AutoModel.from_pretrained(byt5_model_name).to(DEVICE)
byt5_model.eval()

# Model 2: Vyakyarth (for teacher embeddings)
vyakyarth_model = SentenceTransformer("krutrim-ai-labs/vyakyarth")
vyakyarth_model.to(DEVICE)


# === Step 2: Load Sentences and Generate Pre-computed Embeddings in Batches ===

# Load all sentences from your file first. This list will serve as our "metadata".
sn_file = "dev.sn"
sentences = []
try:
    with open(sn_file, "r", encoding="utf-8") as f:
        for line in f:
            stripped = line.strip()
            if stripped:
                sentence = stripped.split("\t")[0]
                sentences.append(sentence)
    print(f"Loaded {len(sentences)} sentences from file.")
except FileNotFoundError:
    print(f"Error: The file '{sn_file}' was not found.")
    print("Using dummy sentences for demonstration purposes.")
    sentences = [
        "अहम् आपणम् गच्छामि।", "सः पुस्तकम् पठति।", "बालकाः कन्दुकेन क्रीडन्ति।", "सूर्यः पूर्वस्याम् दिशि उदेति।",
        "अहम् आपणम् गच्छामि।", "सः पुस्तकम् पठति।", "बालकाः कन्दुकेन क्रीडन्ति।", "सूर्यः पूर्वस्याम् दिशि उदेति।",
        "अहम् आपणम् गच्छामि।", "सः पुस्तकम् पठति।", "बालकाः कन्दुकेन क्रीडन्ति।", "सूर्यः पूर्वस्याम् दिशि उदेति।",
        "अहम् आपणम् गच्छामि।", "सः पुस्तकम् पठति।", "बालकाः कन्दुकेन क्रीडन्ति।", "सूर्यः पूर्वस्याम् दिशि उदेति।"
    ]

# Limit the number of sentences to process
if MAX_SENTENCES is not None and len(sentences) > MAX_SENTENCES:
    sentences = sentences[:MAX_SENTENCES]
    print(f"Processing the first {len(sentences)} sentences as per MAX_SENTENCES setting.")

# Process in batches to save memory
all_vyakyarth_embeddings = []
all_byt5_embeddings = []

print(f"\nGenerating embeddings in batches of {PROCESSING_BATCH_SIZE} to conserve memory...")

for i in tqdm(range(0, len(sentences), PROCESSING_BATCH_SIZE), desc="Processing All Sentences"):
    batch_sentences = sentences[i:i + PROCESSING_BATCH_SIZE]

    batch_embeddings_vyakyarth = vyakyarth_model.encode(
        batch_sentences, convert_to_tensor=True, show_progress_bar=False, device=DEVICE
    )
    all_vyakyarth_embeddings.append(batch_embeddings_vyakyarth)

    batch_embeddings_byt5_list = []
    with torch.no_grad():
        for sentence in batch_sentences:
            inputs = byt5_tokenizer(sentence, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
            output = byt5_model.encoder(**inputs).last_hidden_state
            attention_mask = inputs['attention_mask']
            mask_expanded = attention_mask.unsqueeze(-1).expand(output.size()).float()
            sum_embeddings = torch.sum(output * mask_expanded, 1)
            sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
            pooled = sum_embeddings / sum_mask
            batch_embeddings_byt5_list.append(pooled)
    all_byt5_embeddings.append(torch.cat(batch_embeddings_byt5_list, dim=0))

embeddings_vyakyarth = torch.cat(all_vyakyarth_embeddings, dim=0)
embeddings_byt5 = torch.cat(all_byt5_embeddings, dim=0)

print("\nFinished generating all embeddings.")
print(f"Shape of Vyakyarth embeddings: {embeddings_vyakyarth.shape}")
print(f"Shape of ByT5 embeddings: {embeddings_byt5.shape}")

embeddings_vyakyarth_cpu = embeddings_vyakyarth.cpu()
embeddings_byt5_cpu = embeddings_byt5.cpu()


# === Step 3: Define the Distillation Model ===
class DistillationModel(nn.Module):
    def __init__(self, student_embedding_dim, projection_dim):
        super().__init__()
        self.projection_head = nn.Sequential(
            nn.Linear(student_embedding_dim, student_embedding_dim),
            nn.ReLU(),
            nn.Linear(student_embedding_dim, projection_dim)
        )
    def forward(self, student_embedding):
        return self.projection_head(student_embedding)

# === Step 4: Create a Custom Dataset and DataLoader ===
class DistillationDataset(Dataset):
    def __init__(self, student_embeddings, teacher_embeddings):
        self.student_embeddings = student_embeddings
        self.teacher_embeddings = teacher_embeddings
    def __len__(self):
        return len(self.student_embeddings)
    def __getitem__(self, idx):
        return self.student_embeddings[idx], self.teacher_embeddings[idx]

# === Step 5: Set Up and Run the Training Loop ===
byt5_embedding_dim = embeddings_byt5.shape[1]
vyakyarth_embedding_dim = embeddings_vyakyarth.shape[1]

student_distillation_model = DistillationModel(
    student_embedding_dim=byt5_embedding_dim,
    projection_dim=vyakyarth_embedding_dim
).to(DEVICE)

train_dataset = DistillationDataset(embeddings_byt5_cpu, embeddings_vyakyarth_cpu)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

loss_function = nn.MSELoss()
optimizer = optim.AdamW(student_distillation_model.parameters(), lr=LEARNING_RATE)

print("\nStarting knowledge distillation training...")
student_distillation_model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for student_embs, teacher_embs in progress_bar:
        student_embs, teacher_embs = student_embs.to(DEVICE), teacher_embs.to(DEVICE)
        optimizer.zero_grad()
        distilled_embeddings = student_distillation_model(student_embs)
        loss = loss_function(distilled_embeddings, teacher_embs)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1}/{EPOCHS} - Average Loss: {avg_loss:.4f}")
print("\nTraining complete!")
student_distillation_model.eval() # Set model to evaluation mode for the next steps

# === Step 6: (Optional) Save the trained model ===
# You can save the state of your trained projection head for later use
torch.save(student_distillation_model.state_dict(), 'distilled_model_state.bin')
print("Trained distillation model state saved to 'distilled_model_state.bin'")


# === Step 7: Compare Sentences by Generating Embeddings On-The-Fly ===
def get_distilled_embeddings_for_new_sentences(sentences_list, base_tokenizer, base_model, distillation_model):
    """
    Generates final, distilled embeddings for a list of new sentences.
    """
    final_embeddings = []
    distillation_model.eval() # Ensure model is in eval mode

    with torch.no_grad():
        for sentence in tqdm(sentences_list, desc="Generating New Embeddings"):
            # Step 1: Get the base embedding from ByT5
            inputs = base_tokenizer(str(sentence), return_tensors="pt", padding=True, truncation=True).to(DEVICE)
            output = base_model.encoder(**inputs).last_hidden_state
            attention_mask = inputs['attention_mask']
            mask_expanded = attention_mask.unsqueeze(-1).expand(output.size()).float()
            sum_embeddings = torch.sum(output * mask_expanded, 1)
            sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
            base_embedding = sum_embeddings / sum_mask

            # Step 2: Pass the base embedding through the trained distillation model
            distilled_embedding = distillation_model(base_embedding)
            final_embeddings.append(distilled_embedding.cpu().numpy())

    return np.vstack(final_embeddings)

def compare_sentences_on_the_fly(input_file, col1, col2, output_file, base_tokenizer, base_model, distillation_model):
    """
    Reads a CSV, generates distilled embeddings on-the-fly for two columns,
    calculates cosine similarity, and saves the result.
    """
    print(f"\nProcessing file for on-the-fly similarity comparison: {input_file}")
    try:
        if input_file.endswith('.csv'):
            df = pd.read_csv(input_file)
        elif input_file.endswith('.xlsx'):
            df = pd.read_excel(input_file)
        else:
            print(f"Error: Unsupported file format for {input_file}. Please use .csv or .xlsx.")
            return
    except FileNotFoundError:
        print(f"Error: Input file not found at {input_file}")
        return

    if not all(c in df.columns for c in [col1, col2]):
        print(f"Error: One or both columns ('{col1}', '{col2}') not found in the file.")
        return

    # Get sentences from the dataframe
    sentences1 = df[col1].tolist()
    sentences2 = df[col2].tolist()

    # Generate distilled embeddings for both columns on the fly
    print(f"\nGenerating embeddings for sentences in '{col1}'...")
    embeddings1 = get_distilled_embeddings_for_new_sentences(sentences1, base_tokenizer, base_model, distillation_model)

    print(f"\nGenerating embeddings for sentences in '{col2}'...")
    embeddings2 = get_distilled_embeddings_for_new_sentences(sentences2, base_tokenizer, base_model, distillation_model)

    # Calculate cosine similarity
    print("\nCalculating cosine similarity scores...")
    # We calculate the dot product for each pair (i, i)
    scores = [cosine_similarity(e1.reshape(1, -1), e2.reshape(1, -1))[0, 0] for e1, e2 in zip(embeddings1, embeddings2)]
    print("---")
    print(scores)
    df['combined_score'] = scores

    # Save to the desired format
    if output_file.endswith('.csv'):
        df.to_csv(output_file, index=False)
    elif output_file.endswith('.xlsx'):
        df.to_excel(output_file, index=False)

    print(f"\n🚀 All done! On-the-fly results have been saved to '{output_file}'.")


    return  df['combined_score']

Using device: cpu
Loading models...
Loaded 6148 sentences from file.
Processing the first 50 sentences as per MAX_SENTENCES setting.

Generating embeddings in batches of 10 to conserve memory...


Processing All Sentences:   0%|          | 0/5 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Processing All Sentences: 100%|██████████| 5/5 [04:24<00:00, 52.91s/it]



Finished generating all embeddings.
Shape of Vyakyarth embeddings: torch.Size([50, 768])
Shape of ByT5 embeddings: torch.Size([50, 1536])

Starting knowledge distillation training...


Epoch 1/10: 100%|██████████| 4/4 [00:00<00:00, 21.06it/s, loss=0.0733]


Epoch 1/10 - Average Loss: 0.0829


...

Epoch 8/10 - Average Loss: 0.0496


Epoch 9/10: 100%|██████████| 4/4 [00:00<00:00, 16.43it/s, loss=0.0469]


Epoch 9/10 - Average Loss: 0.0476


Epoch 10/10: 100%|██████████| 4/4 [00:00<00:00, 17.86it/s, loss=0.0469]


Epoch 10/10 - Average Loss: 0.0458

Training complete!
Trained distillation model state saved to 'distilled_model_state.bin'


In [14]:
# --- Execute the on-the-fly similarity calculation ---
input_csv_path = "Similarity_score_baseine.xlsx"
output_csv_path = "similarity_results_on_the_fly.xlxs"
column1_name = "Base Sanskrit"
column2_name = "Most Similar Sanskrit Senetnce ( Retrived from ByT5 & Indicbert)"

scire =compare_sentences_on_the_fly(
    input_file=input_csv_path,
    col1=column1_name,
    col2=column2_name,
    output_file=output_csv_path,
    base_tokenizer=byt5_tokenizer,
    base_model=byt5_model,
    distillation_model=student_distillation_model
)


Processing file for on-the-fly similarity comparison: Similarity_score_baseine.xlsx

Generating embeddings for sentences in 'Base Sanskrit'...


Generating New Embeddings: 100%|██████████| 250/250 [18:44<00:00,  4.50s/it]



Generating embeddings for sentences in 'Most Similar Sanskrit Senetnce ( Retrived from ByT5 & Indicbert)'...


Generating New Embeddings: 100%|██████████| 250/250 [18:36<00:00,  4.46s/it]


Calculating cosine similarity scores...
---
[np.float32(0.9999558), np.float32(0.9999658), np.float32(0.9999684), np.float32(0.9999634), np.float32(0.9999755), np.float32(0.9999728), np.float32(0.999977), np.float32(0.99995375), np.float32(0.99996746), np.float32(0.9999584), np.float32(0.999985), np.float32(0.99998295), np.float32(0.9999746), np.float32(0.99997604), np.float32(0.9999843), np.float32(0.9999764), np.float32(0.9999651), np.float32(0.9999593), np.float32(0.999969), np.float32(0.99997747), np.float32(0.99997985), np.float32(0.99997175), np.float32(0.99996054), np.float32(0.9999807), np.float32(0.99997944), np.float32(0.99997926), np.float32(0.99997413), np.float32(0.9999726), np.float32(0.99997103), np.float32(0.9999693), np.float32(0.99996483), np.float32(0.9999729), np.float32(0.99996316), np.float32(0.99997914), np.float32(0.9999713), np.float32(0.9999654), np.float32(0.99997336), np.float32(0.99996746), np.float32(0.9999727), np.float32(0.9999727), np.float32(0.9999577

In [12]:
scire


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from tqdm import tqdm # For progress bars

# === Configuration ===
# Set to True if a GPU is available for training, otherwise False
USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")
os.environ["CUDA_VISIBLE_DEVICES"] = "0" if USE_CUDA else ""

# Training Hyperparameters
EPOCHS = 10
BATCH_SIZE = 16
LEARNING_RATE = 1e-4

print(f"Using device: {DEVICE}")


# === Step 2: Load Sentences and Generate Pre-computed Embeddings ===

# Load sentences from your file
sn_file = "dev.sn"
sentences = []
try:
    with open(sn_file, "r", encoding="utf-8") as f:
        for line in f:
            stripped = line.strip()
            if stripped:
                sentence = stripped.split("\t")[0]
                sentences.append(sentence)
except FileNotFoundError:
    print(f"Error: The file '{sn_file}' was not found.")
    print("Using dummy sentences for demonstration purposes.")
    sentences = [
        "अहम् आपणम् गच्छामि।",
        "सः पुस्तकम् पठति।",
        "बालकाः कन्दुकेन क्रीडन्ति।",
        "सूर्यः पूर्वस्याम् दिशि उदेति।"
    ]

# 1. Generate Teacher Embeddings from Vyakyarth
print("\nGenerating target embeddings from Vyakyarth (teacher)...")
embeddings_vyakyarth = vyakyarth_model.encode(
    sentences,
    convert_to_tensor=True,
    show_progress_bar=True,
    device=DEVICE
)

# 2. Generate Student Embeddings from Sanskrit-ByT5
print("\nGenerating initial embeddings from Sanskrit-ByT5 (student)...")
embeddings_byt5_list = []
with torch.no_grad():
    for sentence in tqdm(sentences, desc="ByT5 Encoding"):
        inputs = byt5_tokenizer(sentence, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
        output = byt5_model.encoder(**inputs).last_hidden_state

        # Correct mean pooling with attention mask
        attention_mask = inputs['attention_mask']
        mask_expanded = attention_mask.unsqueeze(-1).expand(output.size()).float()
        sum_embeddings = torch.sum(output * mask_expanded, 1)
        sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
        pooled = sum_embeddings / sum_mask

        embeddings_byt5_list.append(pooled)

embeddings_byt5 = torch.cat(embeddings_byt5_list, dim=0)

# Move embeddings to CPU for use in the dataset
embeddings_vyakyarth_cpu = embeddings_vyakyarth.cpu()
embeddings_byt5_cpu = embeddings_byt5.cpu()


# === Step 3: Define the Distillation Model ===

class DistillationModel(nn.Module):
    """
    This model does NOT contain the original encoder. It only contains
    a projection head that takes pre-computed student embeddings as input
    and learns to map them to the teacher's embedding space.
    """
    def __init__(self, student_embedding_dim, projection_dim):
        super().__init__()
        # This simple network will be trained to perform the mapping.
        self.projection_head = nn.Sequential(
            nn.Linear(student_embedding_dim, student_embedding_dim),
            nn.ReLU(),
            nn.Linear(student_embedding_dim, projection_dim)
        )

    def forward(self, student_embedding):
        # Pass the pre-computed student embedding through the projection head
        distilled_embedding = self.projection_head(student_embedding)
        return distilled_embedding

# === Step 4: Create a Custom Dataset and DataLoader ===

class DistillationDataset(Dataset):
    """
    A PyTorch dataset that provides pairs of pre-computed
    student and teacher embeddings.
    """
    def __init__(self, student_embeddings, teacher_embeddings):
        self.student_embeddings = student_embeddings
        self.teacher_embeddings = teacher_embeddings

    def __len__(self):
        return len(self.student_embeddings)

    def __getitem__(self, idx):
        return self.student_embeddings[idx], self.teacher_embeddings[idx]

# === Step 5: Set Up and Run the Training Loop ===

# Get the embedding dimensions for the model
byt5_embedding_dim = embeddings_byt5.shape[1]
vyakyarth_embedding_dim = embeddings_vyakyarth.shape[1]

# Instantiate our student model
student_distillation_model = DistillationModel(
    student_embedding_dim=byt5_embedding_dim,
    projection_dim=vyakyarth_embedding_dim
).to(DEVICE)

# Create the dataset and dataloader using the pre-computed embeddings
train_dataset = DistillationDataset(embeddings_byt5_cpu, embeddings_vyakyarth_cpu)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Define the loss function and optimizer
loss_function = nn.MSELoss()
optimizer = optim.AdamW(student_distillation_model.parameters(), lr=LEARNING_RATE)

print("\nStarting knowledge distillation training...")
student_distillation_model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for student_embs, teacher_embs in progress_bar:
        # Move data to the selected device
        student_embs = student_embs.to(DEVICE)
        teacher_embs = teacher_embs.to(DEVICE)

        optimizer.zero_grad()

        # Forward pass with the pre-computed student embedding
        distilled_embeddings = student_distillation_model(student_embs)

        loss = loss_function(distilled_embeddings, teacher_embs)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1}/{EPOCHS} - Average Loss: {avg_loss:.4f}")

print("\nTraining complete!")

# === Step 6: Generate New Embeddings with the Trained Model ===

print("\nGenerating new, distilled embeddings for all sentences...")
student_distillation_model.eval()
new_combined_embeddings = []

# Use a dataloader for efficient batch processing
inference_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)

with torch.no_grad():
    for student_embs, _ in tqdm(inference_dataloader, desc="Generating Embeddings"):
        student_embs = student_embs.to(DEVICE)

        distilled_embeddings = student_distillation_model(student_embs)
        new_combined_embeddings.extend(distilled_embeddings.cpu().numpy())

new_combined_embeddings = np.array(new_combined_embeddings)

print(f"\nSuccessfully generated {len(new_combined_embeddings)} new embeddings.")
print(f"Shape of the new embedding matrix: {new_combined_embeddings.shape}")

# You can now save or use `new_combined_embeddings` for downstream tasks.
# np.save("distilled_sanskrit_embeddings.npy", new_combined_embeddings)
# print("New embeddings saved to distilled_sanskrit_embeddings.npy")


In [2]:
from google.colab import files
uploaded = files.upload()

Saving test.en to test.en


In [9]:
import pandas as pd
import os # Import the 'os' module to check the directory

# --- 1. Setup ---
english_file = 'test.en'
sanskrit_file = 'test.sn'
output_excel_file = 'krishna_corpus_parsed.xlsx'
search_word = 'krishna'

# --- 2. Diagnostic Checks ---
print("--- Starting Diagnostics ---")
print(f"Current Working Directory: {os.getcwd()}")
print(f"Checking for file: '{english_file}' -> Exists: {os.path.exists(english_file)}")
print(f"Checking for file: '{sanskrit_file}' -> Exists: {os.path.exists(sanskrit_file)}")
print("--------------------------\n")

# --- 3. Process Files and Extract Data ---
found_pairs = []
loop_was_entered = False # A flag to check if we ever enter the loop

try:
    print("Attempting to open files...")
    with open(english_file, 'r', encoding='utf-8') as f_en, \
         open(sanskrit_file, 'r', encoding='utf-8') as f_sn:

        print("Files opened successfully. Reading lines...")
        for en_line, sn_line in zip(f_en, f_sn):
            print(en_line)
            loop_was_entered = True # The loop started, so we set the flag

            # Apply your parsing logic to each line
            en_stripped = en_line.strip()
            # This is your print statement for debugging each line
            print(f"Read English line: '{en_stripped}'")
            sn_stripped = sn_line.strip()

            if en_stripped and sn_stripped:
                # Get the sentence part (before the first tab)
                en_sentence = en_stripped.split("\t")[0]
                sn_sentence = sn_stripped.split("\t")[0]

                # Perform the case-insensitive search on the extracted sentence
                if search_word.lower() in en_sentence.lower():
                    found_pairs.append({
                        'English': en_sentence,
                        'Sanskrit': sn_sentence
                    })

    # Check the flag after the 'with' block
    if not loop_was_entered:
        print("\n⚠️ WARNING: The file reading loop did not run. Please check if your files contain any text.")

except FileNotFoundError as e:
    print(f"\n❌ ERROR: {e}. Make sure the script is in the same directory as your .en and .sn files.")
    exit()

# --- 4. Create Excel File ---
if found_pairs:
    df = pd.DataFrame(found_pairs)
    df.to_excel(output_excel_file, index=False)

    print(f"\n✅ Successfully created '{output_excel_file}' with {len(df)} matching sentence pairs.")
    print("\nPreview of the data:")
    print(df)
else:
    print(f"\nNo lines matching the search word '{search_word}' were found. No Excel file was created.")

--- Starting Diagnostics ---
Current Working Directory: /content
Checking for file: 'test.en' -> Exists: True
Checking for file: 'test.sn' -> Exists: True
--------------------------

Attempting to open files...
Files opened successfully. Reading lines...
Rama went to the forest.	meta_en_1

Read English line: 'Rama went to the forest.	meta_en_1'
The great Krishna spoke the Gita.	meta_en_2

Read English line: 'The great Krishna spoke the Gita.	meta_en_2'
This is another sentence.	meta_en_3

Read English line: 'This is another sentence.	meta_en_3'
Arjuna listened to Krishna's words.	meta_en_4

Read English line: 'Arjuna listened to Krishna's words.	meta_en_4'

✅ Successfully created 'krishna_corpus_parsed.xlsx' with 2 matching sentence pairs.

Preview of the data:
                               English                          Sanskrit
0    The great Krishna spoke the Gita.        महान् कृष्णः गीताम् अवदत्।
1  Arjuna listened to Krishna's words.  अर्जुनः कृष्णस्य वचनं श्रुतवान्।


In [7]:
df

,English,Sanskrit
0,The great Krishna spoke the Gita.,महान् कृष्णः गीताम् अवदत्।
1,Arjuna listened to Krishna's words.,अर्जुनः कृष्णस्य वचनं श्रुतवान्।


In [17]:
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

In [18]:
import pandas as pd
import os

# --- 1. Setup ---
english_file = 'test.en'
sanskrit_file = 'test.sn'
output_excel_file = 'character_corpus_debug.xlsx'
search_words = ['krishna', 'krsna', 'arjuna', 'bheema', 'bhima']

# --- 2. Line Count Diagnostic ---
def count_lines(filename):
    """Helper function to count lines in a file."""
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            return sum(1 for line in f)
    except FileNotFoundError:
        return 0

print("--- File Line Count Diagnostics ---")
en_lines = count_lines(english_file)
sn_lines = count_lines(sanskrit_file)

print(f"Lines found in '{english_file}': {en_lines}")
print(f"Lines found in '{sanskrit_file}': {sn_lines}")

if en_lines != sn_lines:
    print("\n⚠️ WARNING: The files have a different number of lines.")
    print(f"The script will only process the first {min(en_lines, sn_lines)} lines, up to the end of the shortest file.")
print("-----------------------------------\n")


# --- 3. Process Files and Extract Data ---
found_pairs = []

try:
    with open(english_file, 'r', encoding='utf-8') as f_en, \
         open(sanskrit_file, 'r', encoding='utf-8') as f_sn:

        for i, (en_line, sn_line) in enumerate(zip(f_en, f_sn)):
            en_stripped = en_line.strip()
            sn_stripped = sn_line.strip()

            if en_stripped and sn_stripped:
                en_sentence = en_stripped.split("\t")[0]
                sn_sentence = sn_stripped.split("\t")[0]

                if any(word in en_sentence.lower() for word in search_words):
                    found_pairs.append({
                        'English': en_sentence,
                        'Sanskrit': sn_sentence
                    })

except FileNotFoundError:
    print(f"Error: Make sure both '{english_file}' and '{sanskrit_file}' exist in the same directory.")
    exit()

# --- 4. Create Excel File ---
if found_pairs:
    df = pd.DataFrame(found_pairs)
    df.to_excel(output_excel_file, index=False)
    print(f"✅ Successfully created '{output_excel_file}' with {len(df)} matching sentence pairs.")
else:
    print(f"None of the specified words were found within the first {min(en_lines, sn_lines)} lines. No Excel file was created.")

--- File Line Count Diagnostics ---
Lines found in 'test.en': 11721
Lines found in 'test.sn': 11721
-----------------------------------

✅ Successfully created 'character_corpus_debug.xlsx' with 1083 matching sentence pairs.


In [19]:
df['Compare_Sanskrit'] = df['Sanskrit'].iloc[::-1].values

# Reverse the 'English' column and add it as the 4th column
df['Compare_English'] = df['English'].iloc[::-1].values

In [20]:
df

,English,Sanskrit,Compare_Sanskrit,Compare_English
0,"Ascending the sky by the steps of clouds, one ...",शक्यमम्बरमारुह्य मेघसोपानपंक्तिभिः। कुटजार्जुन...,तस्य तद् भाषितं श्रुत्वा त्वरमाणो धनंजयः। वायव...,"Hearing those words of his (Krishna's), Dhanan..."
1,"This hill, having blown Arjunas and Ketakas an...",एष फुल्लार्जुनः शैलः केतकैरभिवासितः। सुग्रीव इ...,ततः प्रसिष्विदे कृष्णः खिन्नश्चार्जुनमब्रवीत्।...,"Then Krishna greatly perspiring, addressed Arj..."
2,It appears like to a drinking-place covered wi...,कदम्बसर्जार्जुनकन्दलाढ्या वनान्तभूमिर्मधुवारिप...,ततस्ते लब्धलक्षत्वादन्योन्यमभिचुकुशुः। हतौ कृष...,"Then finding their arrows strike the aim, they..."
3,The elephants are ranging in this charming for...,प्रमत्तसंनादितबर्हिणानि सशक्रगोपाकुलशाद्वलानि।...,न ध्वजो नार्जुनस्तत्र न रथो न च केशवः। प्रत्यद...,"Thereupon, neither Arjuna, nor his car, nor Ke..."
4,"There appear beautifully on the hills, Ankolas...",अङ्कोलाश्च कुरण्टाश्च चूर्णकाः परिभद्रकाः। चूत...,अयमर्जुनोऽयं गोविन्द इमौ पाण्डवयादवौ। इति ब्रु...,"'This is Arjuna, this is Govinda, these two ar..."
...,...,...,...,...
1078,"'This is Arjuna, this is Govinda, these two ar...",अयमर्जुनोऽयं गोविन्द इमौ पाण्डवयादवौ। इति ब्रु...,अङ्कोलाश्च कुरण्टाश्च चूर्णकाः परिभद्रकाः। चूत...,"There appear beautifully on the hills, Ankolas..."
1079,"Thereupon, neither Arjuna, nor his car, nor Ke...",न ध्वजो नार्जुनस्तत्र न रथो न च केशवः। प्रत्यद...,प्रमत्तसंनादितबर्हिणानि सशक्रगोपाकुलशाद्वलानि।...,The elephants are ranging in this charming for...
1080,"Then finding their arrows strike the aim, they...",ततस्ते लब्धलक्षत्वादन्योन्यमभिचुकुशुः। हतौ कृष...,कदम्बसर्जार्जुनकन्दलाढ्या वनान्तभूमिर्मधुवारिप...,It appears like to a drinking-place covered wi...
1081,"Then Krishna greatly perspiring, addressed Arj...",ततः प्रसिष्विदे कृष्णः खिन्नश्चार्जुनमब्रवीत्।...,एष फुल्लार्जुनः शैलः केतकैरभिवासितः। सुग्रीव इ...,"This hill, having blown Arjunas and Ketakas an..."


In [23]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

output_excel_file = "IndicBert_result.xlsx"
# --- 1. Load Model and Tokenizer ---
print("Loading ai4bharat/indic-bert model...")
model_name = "ai4bharat/indic-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()  # Set the model to evaluation mode

# Use CPU for this task
device = torch.device("cpu")
model.to(device)
print("✅ Model loaded successfully.")

# --- 2. Embedding Helper Function ---
def get_embedding(sentence, model, tokenizer):
    """Generates an embedding for a single sentence using the provided model."""
    # Tokenize the sentence and move to the device
    inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)

    # Get model output without calculating gradients
    with torch.no_grad():
        output = model(**inputs).last_hidden_state

    # Perform mean pooling to get a single sentence vector
    # We take the mean of the token embeddings across the sequence length dimension (dim=1)
    sentence_embedding = output.mean(dim=1)

    # Return the embedding as a NumPy array
    return sentence_embedding.cpu().numpy()



# Limit the DataFrame to the first 3 rows as requested
df_subset = df.head(1000).copy() # Use .copy() to avoid SettingWithCopyWarning

similarity_scores = []
print("\nCalculating similarities for the first 3 rows...")

# Iterate through each of the 3 rows
for index, row in df_subset.iterrows():
    # Get the sentences from the two columns
    sentence1 = row['Sanskrit']
    sentence2 = row['Compare_Sanskrit']

    # Generate embeddings for both sentences
    embedding1 = get_embedding(sentence1, model, tokenizer)
    embedding2 = get_embedding(sentence2, model, tokenizer)

    # Calculate cosine similarity and get the single score value
    score = cosine_similarity(embedding1, embedding2)[0][0]
    similarity_scores.append(score)

# Add the scores as a new column in our subset DataFrame
df_subset['IndicBERT_Similarity'] = similarity_scores

# --- 4. Display Final Result ---
print("\n--- Comparison Results ---")
print(df_subset.head(3))
df_subset.to_excel(output_excel_file, index=False)

Loading ai4bharat/indic-bert model...
✅ Model loaded successfully.

Calculating similarities for the first 3 rows...

--- Comparison Results ---
                                             English  \
0  Ascending the sky by the steps of clouds, one ...   
1  This hill, having blown Arjunas and Ketakas an...   
2  It appears like to a drinking-place covered wi...   

                                            Sanskrit  \
0  शक्यमम्बरमारुह्य मेघसोपानपंक्तिभिः। कुटजार्जुन...   
1  एष फुल्लार्जुनः शैलः केतकैरभिवासितः। सुग्रीव इ...   
2  कदम्बसर्जार्जुनकन्दलाढ्या वनान्तभूमिर्मधुवारिप...   

                                    Compare_Sanskrit  \
0  तस्य तद् भाषितं श्रुत्वा त्वरमाणो धनंजयः। वायव...   
1  ततः प्रसिष्विदे कृष्णः खिन्नश्चार्जुनमब्रवीत्।...   
2  ततस्ते लब्धलक्षत्वादन्योन्यमभिचुकुशुः। हतौ कृष...   

                                     Compare_English  IndicBERT_Similarity  
0  Hearing those words of his (Krishna's), Dhanan...              0.913567  
1  Then Krishna greatly pe

In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

# --- 1. Load Model and Tokenizer (MODIFIED SECTION) ---
print("Loading chronbmm/sanskrit-byt5-dp model...")
model_name = "chronbmm/sanskrit-byt5-dp" # <-- Changed model name
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()  # Set the model to evaluation mode

# Use CPU for this task
device = torch.device("cpu")
model.to(device)
print("✅ Model loaded successfully.")


# --- 2. Embedding Helper Function (No changes needed) ---
def get_embedding(sentence, model, tokenizer):
    """Generates an embedding for a single sentence using the provided model."""
    # Tokenize the sentence and move to the device
    inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)

    # Get model output without calculating gradients
    with torch.no_grad():
        output = model.encoder(**inputs).last_hidden_state # Use model.encoder for ByT5

    # Perform mean pooling to get a single sentence vector
    sentence_embedding = output.mean(dim=1)

    # Return the embedding as a NumPy array
    return sentence_embedding.cpu().numpy()



output_excel_file = "ByT5_result.xlsx" # <-- Changed output file name


# Limit the DataFrame to the first 1000 rows
df_subset_byt5 = df.head(1000).copy()

similarity_scores = []
print(f"\nCalculating similarities for {len(df_subset_byt5)} rows...")

# Iterate through each row
for index, row in df_subset_byt5.iterrows():
    # Get the sentences from the two columns
    sentence1 = row['Sanskrit']
    sentence2 = row['Compare_Sanskrit']

    # Generate embeddings for both sentences
    embedding1 = get_embedding(sentence1, model, tokenizer)
    embedding2 = get_embedding(sentence2, model, tokenizer)

    # Calculate cosine similarity
    score = cosine_similarity(embedding1, embedding2)[0][0]
    similarity_scores.append(score)

# Add the scores as a new column in our subset DataFrame
df_subset_byt5['ByT5_Similarity'] = similarity_scores # <-- Changed new column name

# --- 4. Display and Save Final Result ---
print("\n--- Comparison Results ---")
print(df_subset_byt5.head())
df_subset_byt5.to_excel(output_excel_file, index=False)
print(f"\n✅ Analysis complete. Results saved to '{output_excel_file}'.")

Loading chronbmm/sanskrit-byt5-dp model...


tokenizer_config.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/866 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

✅ Model loaded successfully.

Calculating similarities for 1000 rows...


In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# --- 1. Load Model (MODIFIED SECTION) ---
print("Loading krutrim-ai-labs/vyakyarth model...")
# The SentenceTransformer library simplifies loading and embedding
model = SentenceTransformer("krutrim-ai-labs/vyakyarth")
print("✅ Model loaded successfully.")


# --- 2. Load Data and Process ---
# In your code, you would load your actual Excel file first
# df = pd.read_excel("your_input_file.xlsx")


output_excel_file = "Vyakyarth_result.xlsx" # <-- Changed output file name


# Limit the DataFrame to the first 1000 rows
df_subset = df.head(1000).copy()

similarity_scores = []
print(f"\nCalculating similarities for {len(df_subset)} rows using Vyakyarth...")

# Iterate through each row
for index, row in df_subset.iterrows():
    # Get the sentences from the two columns
    sentence1 = row['Sanskrit']
    sentence2 = row['Compare_Sanskrit']

    # --- Generate embeddings for both sentences (MODIFIED SECTION) ---
    # The encode method handles tokenization and embedding in one step
    # We encode them together for efficiency
    embeddings = model.encode([sentence1, sentence2], convert_to_numpy=True)
    embedding1 = embeddings[0].reshape(1, -1)
    embedding2 = embeddings[1].reshape(1, -1)

    # Calculate cosine similarity
    score = cosine_similarity(embedding1, embedding2)[0][0]
    similarity_scores.append(score)

# Add the scores as a new column in our subset DataFrame
df_subset['Vyakyarth_Similarity'] = similarity_scores # <-- Changed new column name

# --- 3. Display and Save Final Result ---
print("\n--- Comparison Results ---")
print(df_subset.head())
df_subset.to_excel(output_excel_file, index=False)
print(f"\n✅ Analysis complete. Results saved to '{output_excel_file}'.")

In [ ]:
# --- 3. Prepare Data and Process ---
# Assume 'df' is your full DataFrame with 2669 rows.
# For demonstration, we create a sample one.
num_rows = 2669

output_excel_file = 'top1000_similarity_analysis1.xlsx'

# --- Select the top 500 rows for processing ---
df_subset = df.head(1000).copy()

similarity_scores = []
print(f"\nCalculating similarities for the top {len(df_subset)} rows...")

#
# --- THIS IS THE CRITICAL CHANGE ---
# Ensure the loop iterates over df_subset, not the original df.
#
for index, row in df_subset.iterrows():
    sentence1 = row['Sanskrit']
    sentence2 = row['Reversed Sanskrit']

    embedding1 = get_embedding(sentence1, model, tokenizer)
    embedding2 = get_embedding(sentence2, model, tokenizer)

    score = cosine_similarity(embedding1, embedding2)[0][0]
    similarity_scores.append(score)

# This assignment will now work because len(similarity_scores) will be 500
df_subset['sanskrit sscore'] = similarity_scores

# --- 4. Display and Save Final Result ---
print("\n--- Final Comparison Results (Top 500 Rows) ---")
print(df_subset)

# Save the 500-row DataFrame with the new scores to Excel
df_subset.to_excel(output_excel_file, index=False)
print(f"\n✅ Analysis of top 500 rows complete. Results saved to '{output_excel_file}'.")

In [10]:
df.to_excel(output_excel_file, index=False)